In [ ]:
"""
Endurance testing with periodic 3PP until failure

"""
import numpy as np
from pulse_3pp import run_3pp
from bnc765_driver import BNC765
import time
from datetime import datetime
from pathlib import Path
import json

def run_endurance_until_failure(
    stress_amplitude,           # Voltage for stress pulses (V) - PARAMETER
    stress_frequency_hz,        # Frequency of stress pulses (Hz) - PARAMETER
    stress_pulse_width_ns,      # Width of stress pulses (ns)
    test_3pp_params,            # Dict of 3PP parameters (CONSTANT)
    save_directory,             # Where to save results
    failure_criterion=None,     # Optional: function to detect failure from 3PP data
    max_cycles=1e11,            # Default to 100 billion cycles
    checkpoint_interval=1000,   # Save progress every N cycles
    verbose=True
):
    """
    Run endurance test until capacitor fails

    Args:
        stress_amplitude: Amplitude of alternating stress pulses (V)
        stress_frequency_hz: Frequency of stress cycling (Hz)
        stress_pulse_width_ns: Stress pulse width (ns)
        test_3pp_params: Dictionary with 3PP parameters (kept constant)
        save_directory: Directory to save all data
        failure_criterion: Optional function(data) -> bool to detect failure
        max_cycles: Maximum cycles before auto-stop (default 1e11)
        checkpoint_interval: Save checkpoint every N cycles
        verbose: Print progress
    """

    pulser = BNC765("TCPIP::169.254.209.156::INSTR")

    # Calculate decade measurement points up to max_cycles
    measurement_points = []
    decade = 10
    while decade <= max_cycles:
        measurement_points.append(int(decade))
        decade *= 10

    # Calculate stress period from frequency
    stress_period_ns = 1e9 / stress_frequency_hz  # Convert Hz to ns period

    results = {
        'test_parameters': {
            'stress_amplitude': stress_amplitude,
            'stress_frequency_hz': stress_frequency_hz,
            'stress_pulse_width_ns': stress_pulse_width_ns,
            'stress_period_ns': stress_period_ns,
            '3pp_parameters': test_3pp_params,
            'start_time': datetime.now().isoformat()
        },
        'stress_cycles': [],
        'measurement_data': [],
        'timestamps': [],
        'failed': False,
        'failure_cycle': None
    }

    try:
        if verbose:
            print(f"\n{'='*70}")
            print(f"ENDURANCE TEST CONFIGURATION")
            print(f"{'='*70}")
            print(f"Stress amplitude: {stress_amplitude} V")
            print(f"Stress frequency: {stress_frequency_hz} Hz")
            print(f"Stress pulse width: {stress_pulse_width_ns} ns")
            print(f"Stress period: {stress_period_ns:.2f} ns")
            print(f"Maximum cycles: {max_cycles:.2e}")
            print(f"Measurement points: {len(measurement_points)} decades")
            print(f"{'='*70}\n")

        # Setup alternating stress pulses
        setup_stress_pulses(
            pulser,
            amplitude=stress_amplitude,
            pulse_width_ns=stress_pulse_width_ns,
            period_ns=stress_period_ns,
            verbose=verbose
        )

        cycle_count = 0
        next_measurement_idx = 0
        last_checkpoint = 0

        while cycle_count < max_cycles:
            # Determine cycles until next measurement or checkpoint
            if next_measurement_idx < len(measurement_points):
                cycles_to_next = measurement_points[next_measurement_idx] - cycle_count
            else:
                cycles_to_next = checkpoint_interval

            if verbose and (cycle_count % 10000 == 0 or cycle_count in measurement_points):
                elapsed_time = (cycle_count / stress_frequency_hz) / 3600  # hours
                print(f"Cycle: {cycle_count:.2e} | "
                      f"Time: {elapsed_time:.2f} hrs | "
                      f"Next measurement: {measurement_points[next_measurement_idx]:.2e}")

            # Apply stress cycles
            apply_stress_cycles(
                pulser,
                num_cycles=cycles_to_next,
                frequency_hz=stress_frequency_hz
            )
            cycle_count += cycles_to_next

            # Check if we've reached a measurement point
            if next_measurement_idx < len(measurement_points) and \
               cycle_count >= measurement_points[next_measurement_idx]:

                if verbose:
                    print(f"\n{'='*70}")
                    print(f"3PP MEASUREMENT AT {cycle_count:.2e} CYCLES")
                    print(f"{'='*70}")

                # Stop stress pulsing
                pulser.stop()
                time.sleep(1)

                # Run 3PP measurement with CONSTANT parameters
                data = run_3pp(
                    **test_3pp_params,
                    save_directory=f"{save_directory}/cycle_{int(cycle_count):.2e}",
                    auto_trigger=True,
                    save_data=True,
                    save_plot=True,
                    verbose=verbose
                )

                # Store results
                results['stress_cycles'].append(cycle_count)
                results['measurement_data'].append({
                    'cycle': cycle_count,
                    'voltage_max': float(np.max(data['voltage'])),
                    'voltage_min': float(np.min(data['voltage'])),
                    'voltage_mean': float(np.mean(data['voltage'])),
                    'voltage_std': float(np.std(data['voltage']))
                })
                results['timestamps'].append(datetime.now().isoformat())

                # Check for failure if criterion provided
                if failure_criterion is not None:
                    if failure_criterion(data):
                        print(f"\n{'='*70}")
                        print(f"FAILURE DETECTED AT {cycle_count:.2e} CYCLES")
                        print(f"{'='*70}\n")
                        results['failed'] = True
                        results['failure_cycle'] = cycle_count
                        save_endurance_results(results, save_directory)
                        break

                # Save checkpoint
                save_endurance_results(results, save_directory)

                next_measurement_idx += 1

                # Resume stress pulsing
                pulser.start()

            # Periodic checkpoint saving
            if cycle_count - last_checkpoint >= checkpoint_interval:
                save_endurance_results(results, save_directory, is_checkpoint=True)
                last_checkpoint = cycle_count

        # Test completed without failure
        results['test_parameters']['end_time'] = datetime.now().isoformat()
        save_endurance_results(results, save_directory)

        if verbose:
            print(f"\n{'='*70}")
            print(f"ENDURANCE TEST {'FAILED' if results['failed'] else 'COMPLETE'}")
            print(f"Total stress cycles: {cycle_count:.2e}")
            print(f"Measurements taken: {len(results['stress_cycles'])}")
            if results['failed']:
                print(f"Failure at cycle: {results['failure_cycle']:.2e}")
            print(f"{'='*70}\n")

        return results

    except KeyboardInterrupt:
        print("\n\nTest interrupted by user!")
        results['test_parameters']['interrupted'] = True
        results['test_parameters']['end_time'] = datetime.now().isoformat()
        save_endurance_results(results, save_directory)
        return results

    finally:
        pulser.ch1.output_state = False
        pulser.ch2.output_state = False
        pulser.stop()
        pulser.shutdown()


def setup_stress_pulses(pulser, amplitude, pulse_width_ns, period_ns, verbose=False):
    """
    Setup alternating positive/negative stress pulses on two channels
    CH1: Positive pulses (0→+V)
    CH2: Negative pulses (0→-V), offset by half period
    """
    if verbose:
        print("Configuring stress pulse channels...")

    pulser.stop()

    # Channel 1: Positive stress pulses
    pulser.ch1.pulse_mode = 'SIN'
    pulser.ch1.voltage_level = amplitude
    pulser.ch1.voltage_offset = amplitude / 2  # 0V to +V
    pulser.ch1.inverted = False
    pulser.ch1.load_impedance = 50
    pulser.write(f'SOURCE1:PULSE1:WIDTH {pulse_width_ns}E-9')
    pulser.write(f'SOURCE1:PULSE1:DELAY 0')
    pulser.ch1.period = f'{period_ns}ns'
    pulser.ch1.output_state = True

    # Channel 2: Negative stress pulses (offset by half period)
    pulser.ch2.pulse_mode = 'SIN'
    pulser.ch2.voltage_level = amplitude
    pulser.ch2.voltage_offset = -amplitude / 2  # 0V to -V
    pulser.ch2.inverted = False
    pulser.ch2.load_impedance = 50
    pulser.write(f'SOURCE2:PULSE1:WIDTH {pulse_width_ns}E-9')
    pulser.write(f'SOURCE2:PULSE1:DELAY {period_ns/2}E-9')  # Offset by half period
    pulser.ch2.period = f'{period_ns}ns'
    pulser.ch2.output_state = True

    # Continuous mode for stress cycling
    pulser.trigger_mode = 'CONTINUOUS'
    pulser.start()

    if verbose:
        print(f"  CH1: +{amplitude}V pulses")
        print(f"  CH2: -{amplitude}V pulses (offset by {period_ns/2:.1f} ns)")


def apply_stress_cycles(pulser, num_cycles, frequency_hz):
    """
    Apply specified number of stress cycles
    In continuous mode, calculate time and wait
    """
    # Time to apply num_cycles at given frequency
    duration_seconds = num_cycles / frequency_hz
    time.sleep(duration_seconds)


def save_endurance_results(results, directory, is_checkpoint=False):
    """Save endurance test results to JSON file"""
    save_path = Path("../data") / directory
    save_path.mkdir(parents=True, exist_ok=True)

    filename = 'endurance_checkpoint.json' if is_checkpoint else 'endurance_results.json'

    with open(save_path / filename, 'w') as f:
        json.dump(results, f, indent=2)


# Optional: Define failure detection function
def detect_capacitor_failure(data):
    """
    Example failure criterion: Check if signal amplitude has degraded significantly
    Customize this based on your specific failure indicators
    """
    voltage_range = np.max(data['voltage']) - np.min(data['voltage'])

    # Example: Failure if amplitude drops below 50% of expected
    # Adjust threshold based on your device specifications
    expected_amplitude = 0.5  # V (adjust to match your 3PP parameters)
    threshold = 0.5 * expected_amplitude

    if voltage_range < threshold:
        return True  # Failure detected

    return False


# Example usage
if __name__ == "__main__":
    # Define constant 3PP measurement parameters
    constant_3pp_params = {
        'u_amplitude': 0.5,
        'u_to_n_delay': 500.0,
        'nd_amplitude': 0.5,
        'n_to_d_delay': 100.0,
        'polarity': 'npp',
        'pulse_width_ns': 200.0,
        'capture_width_ns': 2000.0,
        'num_averages': 16,
        'vdiv': 0.2
    }

    # Run endurance test with settable amplitude and frequency
    run_endurance_until_failure(
        stress_amplitude=0.3,              # PARAMETER: Stress voltage
        stress_frequency_hz=1e6,           # PARAMETER: 1 MHz stress frequency
        stress_pulse_width_ns=100.0,       # Stress pulse width
        test_3pp_params=constant_3pp_params,  # CONSTANT 3PP parameters
        save_directory='endurance_0p3V_1MHz',
        failure_criterion=detect_capacitor_failure,  # Optional
        max_cycles=1e11,                   # Run up to 100 billion cycles
        verbose=True
    )